# 02 — Pipeline & Inferenz

Phase 3 (Pipeline bauen + erste Inferenz auf 12 Hand-Gold-Anzeigen), Phase 4 (Iterationen A + B — pro Iteration eigener Predictions-Dateiname und Run-Header-Update), Phase 6 (voller Korpus auf 7B + 3B-Kontrast auf euler).

Cheatsheets: `CHEATSHEETS/transformers-konzepte.md` (Modell, Chat-Template, JSON-Parsing), `CHEATSHEETS/gpu-zugang.md` (Spawn, GPU-Wahl, Memory).

## Run-Header

# 02 — Pipeline: Qwen2.5-7B Information Extraction

| Feld | Wert |
|---|---|
| Modell | `Qwen/Qwen2.5-7B-Instruct` |
| Server | gauss |
| GPU-Index | 1 (CUDA_VISIBLE_DEVICES) |
| Datum | 2026-05-11 |
| Korpus | `daten/eigener_korpus.jsonl` (55 Anzeigen, 12 davon Hand-Gold aus Phase 2) |
| Output | `daten/predictions.jsonl` |
| Schema | siehe `SCHEMA.md` (6 Felder, fix) |
| Truncation | Head, ~2000 Zeichen |
| Generation | greedy, max_new_tokens=200 |'

## Phase 3 — Pipeline bauen + Baseline-Inferenz

In [1]:
import json
import re
import time
import os
from pathlib import Path
from transformers import AutoTokenizer, AutoModelForCausalLM
import torch
os.environ["CUDA_VISIBLE_DEVICES"] = "3"   # ggf. anpassen je nach Sitzplatz-Mapping

print(f"CUDA_VISIBLE_DEVICES: {os.environ.get('CUDA_VISIBLE_DEVICES', '(nicht gesetzt)')}")
print(f"PyTorch:    {torch.__version__}")
print(f"CUDA verfügbar: {torch.cuda.is_available()}")
print(f"Sichtbare GPUs: {torch.cuda.device_count()}")
if torch.cuda.is_available():
    print(f"Aktive GPU:  {torch.cuda.current_device()} = {torch.cuda.get_device_name(0)}")

CUDA_VISIBLE_DEVICES: 3
PyTorch:    1.12.1
CUDA verfügbar: True
Sichtbare GPUs: 1
Aktive GPU:  0 = Tesla V100-SXM2-32GB


## Phase 4 — Iteration A

In [2]:
"""
Modell-Load: Qwen2.5-7B-Instruct
- torch_dtype=float32 (V100-Constraint, NICHT fp16/bf16!)
- .to("cuda").eval() (NICHT device_map="auto"!)
- Modell ist auf gauss vorab gecached → kein Download
"""
MODEL_NAME = "Qwen/Qwen2.5-7B-Instruct"

print(f"Lade Modell {MODEL_NAME} …")
t0 = time.time()

tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)
model = AutoModelForCausalLM.from_pretrained(
    MODEL_NAME,
    torch_dtype=torch.float32,
).to("cuda").eval()

ladezeit = time.time() - t0
print(f"✓ Modell geladen in {ladezeit:.1f}s")
print(f"  GPU-Memory belegt: "
      f"{torch.cuda.memory_allocated() / 1024**3:.1f} GB "
      f"von {torch.cuda.get_device_properties(0).total_memory / 1024**3:.1f} GB")

Lade Modell Qwen/Qwen2.5-7B-Instruct …


/opt/conda/envs/torch/lib/python3.10/site-packages/huggingface_hub/file_download.py:949: FutureWarning: `resume_download` is deprecated and will be removed in version 1.0.0. Downloads always resume when possible. If you want to force a new download, use `force_download=True`.
  warnings.warn(


tokenizer_config.json: 0.00B [00:00, ?B/s]

vocab.json: 0.00B [00:00, ?B/s]

merges.txt: 0.00B [00:00, ?B/s]

...

Xet Storage is enabled for this repo, but the 'hf_xet' package is not installed. Falling back to regular HTTP download. For better performance, install the package with: `pip install huggingface_hub[hf_xet]` or `pip install hf_xet`


model-00004-of-00004.safetensors:   0%|          | 0.00/3.56G [00:00<?, ?B/s]

Loading checkpoint shards:   0%|          | 0/4 [00:00<?, ?it/s]

generation_config.json:   0%|          | 0.00/243 [00:00<?, ?B/s]

✓ Modell geladen in 144.7s
  GPU-Memory belegt: 29.4 GB von 31.5 GB


## Phase 4 — Iteration B

In [3]:
"""
Iteration B: Prompt-Klarstellung. 
Truncation zurück auf einfach (head-only, 2000 Zeichen) damit Prompt-Effekt isoliert messbar.
"""
import re

SCHEMA_BESCHREIBUNG = """Du bist ein Assistent, der strukturierte Informationen aus deutschen Stellenanzeigen extrahiert.

Antworte AUSSCHLIESSLICH mit einem JSON-Objekt im folgenden Schema. Kein Vor- oder Nachtext.

SCHEMA (6 Felder, fix):
- "homeoffice": einer von ["ja", "teilweise", "nein", "remote", "nicht_genannt"]
- "vertragsart": einer von ["ausbildung", "festanstellung", "praktikum", "werkstudent", "sonstiges"]
- "erfahrungslevel": einer von ["junior", "mid", "senior", "egal", "nicht_genannt"]
- "gehalt_min_eur": ganze Zahl (int) oder null
- "gehalt_zeitraum": "monat", "jahr" oder null (muss null sein wenn gehalt_min_eur null ist)
- "skills_top3": Liste mit max. 3 technischen Skills/Tools/Sprachen

BEISPIEL-ANTWORT:
{"homeoffice": "teilweise", "vertragsart": "festanstellung", "erfahrungslevel": "mid", "gehalt_min_eur": 55000, "gehalt_zeitraum": "jahr", "skills_top3": ["python", "sql", "tableau"]}

HARTE REGELN (BEFOLGE STRIKT):

1. gehalt_min_eur: Wenn KEINE konkrete Zahl im Text steht, IMMER null. NIEMALS raten, schätzen oder aus Branchenwissen ableiten. 'Attraktive Vergütung' / 'nach Vereinbarung' / 'marktüblich' → null.

2. erfahrungslevel:
   - 'Ausbildung' / 'Auszubildende' → IMMER junior
   - 'erste Berufserfahrung' / 'max. 2 Jahre' / 'Berufseinsteiger' / 'Studienabsolvent' / '0-2 Jahre' → IMMER junior
   - 'Senior' im Titel ODER '5+ Jahre' / '8+ Jahre' / 'Mentoring von Juniors' → senior
   - '2-5 Jahre' / 'mehrjährig' / 'Erfahrung erforderlich' (ohne weitere Spezifikation) → mid
   - egal: NUR wenn explizit 'Junior bis Senior willkommen' o.ä. steht. Niemals als Default.
   - nicht_genannt: NUR wenn Text gar keine Aussage zur Erfahrung macht.

3. homeoffice:
   - 'hybrid' / 'X Tage/Woche Homeoffice' / 'mobiles Arbeiten' / 'Home Office möglich' → teilweise
   - '100% remote' / 'Vollzeit Remote' → remote
   - 'Präsenzpflicht' / 'kein Homeoffice' → nein
   - 'Homeoffice-Möglichkeit' OHNE Details → ja
   - Marketing-Slogans wie 'feels like home', 'familiäres Umfeld' → NICHT als Homeoffice werten. Das ist Werbung.
   - 'flexible Arbeitszeit' bezieht sich auf ZEIT, NICHT auf ORT → NICHT als Homeoffice werten.
   - Nichts dazu im Text → nicht_genannt.

4. skills_top3:
   - NUR konkrete Tools, Sprachen, Frameworks, Plattformen (Beispiele: Python, SQL, Java, Spark, AWS, Azure, scikit-learn, React, Power BI, Tableau, Git).
   - NICHT Methoden oder Konzepte: 'agile', 'machine learning', 'frameworks', 'testen', 'softwareentwicklung'.
   - NICHT Soft Skills: 'Teamfähigkeit', 'Kommunikation'.
   - NICHT Sprachen: 'Deutsch', 'Englisch'.
   - Bei Ausbildungs-Stellen: nur Tools, die explizit als BEWERBUNGSVORAUSSETZUNG genannt sind (z.B. 'Erste Programmierkenntnisse in Java erforderlich'). NICHT Tools, die 'gelernt werden' während der Ausbildung.
   - Wenn keine konkreten Tools im Text → leere Liste [].

5. gehalt_zeitraum: muss null sein, wenn gehalt_min_eur null ist.
"""

MAX_TEXT_LAENGE = 1500


def baue_prompt(anzeigentext: str) -> str:
    """Baut den fertigen Prompt-String mit Chat-Template (Iteration B: head-truncation zurück)."""
    text_kurz = anzeigentext[:MAX_TEXT_LAENGE]
    messages = [
        {"role": "system", "content": SCHEMA_BESCHREIBUNG},
        {"role": "user", "content": f"Extrahiere die Felder aus dieser Stellenanzeige:\n\n{text_kurz}"},
    ]
    return tokenizer.apply_chat_template(
        messages, tokenize=False, add_generation_prompt=True
    )


def parse_json_robust(response: str) -> dict | None:
    """Versucht JSON aus Modell-Output zu extrahieren - auch wenn Prosa drumrum ist."""
    match = re.search(r"\{.*\}", response, re.DOTALL)
    if not match:
        return None
    try:
        return json.loads(match.group(0))
    except json.JSONDecodeError:
        return None


def extrahiere(anzeigentext: str) -> tuple:
    """Eine Anzeige -> (parsed_json_oder_None, raw_response, dauer_in_s)."""
    prompt = baue_prompt(anzeigentext)
    inputs = tokenizer(prompt, return_tensors="pt").to("cuda")

    t0 = time.time()
    with torch.no_grad():
        outputs = model.generate(
            **inputs,
            max_new_tokens=200,
            do_sample=False,                       # Greedy, reproduzierbar
            pad_token_id=tokenizer.eos_token_id,
        )
    dauer = time.time() - t0

    response = tokenizer.decode(
        outputs[0][inputs.input_ids.shape[1]:],
        skip_special_tokens=True,
    )
    parsed = parse_json_robust(response)
    return parsed, response, dauer


print("✓ Iteration B aktiviert: verschärfter Prompt mit harten Regeln")
print(f"  Truncation: Head, {MAX_TEXT_LAENGE} Zeichen (zurück zu Baseline)")
print(f"  Prompt-Länge System: {len(SCHEMA_BESCHREIBUNG)} Zeichen")

✓ Iteration B aktiviert: verschärfter Prompt mit harten Regeln
  Truncation: Head, 1500 Zeichen (zurück zu Baseline)
  Prompt-Länge System: 2894 Zeichen


## Phase 6 — Voller 7B-Run (gauss)

In [4]:
"""
Memory cleanup: leert PyTorch's KV-Cache vor dem großen Loop.
Modell + Tokenizer bleiben erhalten — nur Inferenz-Reste werden freigegeben.
"""
import torch
import gc

# Free up cached memory
torch.cuda.empty_cache()
gc.collect()

# Check
free, total = torch.cuda.mem_get_info(0)
print(f"GPU-Memory nach cleanup:")
print(f"  Frei:    {free/1e9:.1f} GB")
print(f"  Belegt:  {(total-free)/1e9:.1f} GB von {total/1e9:.1f} GB")

GPU-Memory nach cleanup:
  Frei:    1.6 GB
  Belegt:  32.5 GB von 34.1 GB


In [5]:
"""
Test-Lauf an Anzeige 1 (KOSATEC Data Scientist).
Wenn das hier sauber durchläuft, kann die Schleife auf alle 12 starten.
"""
# Korpus laden
korpus = {}
with open("../daten/eigener_korpus.jsonl", encoding="utf-8") as f:
    for line in f:
        a = json.loads(line)
        korpus[a["refnr"]] = a

print(f"Korpus geladen: {len(korpus)} Anzeigen")

# Test-Anzeige
test_refnr = "13644-294294-S"   # KOSATEC Data Scientist
test_anzeige = korpus[test_refnr]
print(f"\nTest-Anzeige: {test_anzeige['titel']} ({test_anzeige['firma']})")
print(f"Textlänge: {len(test_anzeige['text'])} Zeichen (truncated auf {MAX_TEXT_LAENGE})")

# Extraktion
parsed, raw, dauer = extrahiere(test_anzeige["text"])

print(f"\n── Inferenz-Zeit: {dauer:.2f}s ──")
print(f"\n── Raw-Output des Modells ──")
print(raw)
print(f"\n── Geparstes JSON ──")
if parsed:
    print(json.dumps(parsed, indent=2, ensure_ascii=False))
else:
    print("⚠ Kein JSON extrahierbar — siehe Raw-Output")

Korpus geladen: 55 Anzeigen

Test-Anzeige: Data Scientist (m/w/d) (KOSATEC)
Textlänge: 1907 Zeichen (truncated auf 1500)


/opt/conda/envs/torch/lib/python3.10/site-packages/transformers/generation/configuration_utils.py:492: UserWarning: `do_sample` is set to `False`. However, `temperature` is set to `0.7` -- this flag is only used in sample-based generation modes. You should set `do_sample=True` or unset `temperature`.
  warnings.warn(
/opt/conda/envs/torch/lib/python3.10/site-packages/transformers/generation/configuration_utils.py:497: UserWarning: `do_sample` is set to `False`. However, `top_p` is set to `0.8` -- this flag is only used in sample-based generation modes. You should set `do_sample=True` or unset `top_p`.
  warnings.warn(
/opt/conda/envs/torch/lib/python3.10/site-packages/transformers/generation/configuration_utils.py:509: UserWarning: `do_sample` is set to `False`. However, `top_k` is set to `20` -- this flag is only used in sample-based generation modes. You should set `do_sample=True` or unset `top_k`.
  warnings.warn(



── Inferenz-Zeit: 5.39s ──

── Raw-Output des Modells ──
{
  "homeoffice": "nicht_genannt",
  "vertragsart": "festanstellung",
  "erfahrungslevel": "mid",
  "gehalt_min_eur": null,
  "gehalt_zeitraum": null,
  "skills_top3": ["python", "sql", "r"]
}

── Geparstes JSON ──
{
  "homeoffice": "nicht_genannt",
  "vertragsart": "festanstellung",
  "erfahrungslevel": "mid",
  "gehalt_min_eur": null,
  "gehalt_zeitraum": null,
  "skills_top3": [
    "python",
    "sql",
    "r"
  ]
}


## Phase 6 — 3B-Run (euler)

In [6]:
"""
Volle Pipeline: Extraktion über die 12 Hand-Gold-Anzeigen aus Phase 2.
Ergebnis -> daten/predictions.jsonl

Format pro Zeile: JSON-Objekt mit refnr + den 6 Schema-Feldern.
Bei JSON-Parse-Fail: refnr + parse_fail=true, ohne Schema-Felder.
"""
# Dieselben 12 refnrs wie in meine_gold.csv und partner_gold.csv
HAND_GOLD_REFNRS = [
    "13644-294294-S",
    "10001-1002678126-S",
    "10000-1185445644-S",
    "10001-1002026714-S",
    "15719-44156922-55-S",
    "16818-100787927-S",
    "19384-938440348-S",
    "10001-1003030981-S",
    "16818-100814418-S",
    "13644-297872-S",
    "13319-868490/1_607417LS-S",
    "10001-1002616378-S",
]

ausgabe_pfad = Path("../daten/predictions_iterB.jsonl")
ausgabe_pfad.parent.mkdir(exist_ok=True)

print(f"Starte Inferenz über {len(HAND_GOLD_REFNRS)} Anzeigen …\n")

stats = {"ok": 0, "parse_fail": 0, "gesamt_dauer": 0.0}

with ausgabe_pfad.open("w", encoding="utf-8") as f_out:
    for i, refnr in enumerate(HAND_GOLD_REFNRS, 1):
        anzeige = korpus[refnr]
        parsed, raw, dauer = extrahiere(anzeige["text"])
        stats["gesamt_dauer"] += dauer

        if parsed is None:
            stats["parse_fail"] += 1
            output = {"refnr": refnr, "parse_fail": True, "raw_response": raw}
            status = "✗ PARSE-FAIL"
        else:
            stats["ok"] += 1
            output = {"refnr": refnr, **parsed}
            status = "✓"

        f_out.write(json.dumps(output, ensure_ascii=False) + "\n")
        f_out.flush()

        print(f"{status} [{i:>2}/{len(HAND_GOLD_REFNRS)}] {refnr:<28} "
              f"({dauer:.1f}s)  {anzeige['titel'][:40]}")

print(f"\n{'─'*70}")
print(f"✓ Fertig! {stats['ok']} OK, {stats['parse_fail']} Parse-Fails")
print(f"  Gesamt-Dauer: {stats['gesamt_dauer']:.1f}s "
      f"(Ø {stats['gesamt_dauer']/len(HAND_GOLD_REFNRS):.1f}s pro Anzeige)")
print(f"  Output: {ausgabe_pfad}")

Starte Inferenz über 12 Anzeigen …

✓ [ 1/12] 13644-294294-S               (5.0s)  Data Scientist (m/w/d)
✓ [ 2/12] 10001-1002678126-S           (5.4s)  Data Scientist (m/w/d)
✓ [ 3/12] 10000-1185445644-S           (4.3s)  Data Scientist (m/w/d)
✓ [ 4/12] 10001-1002026714-S           (4.7s)  Fachinformatiker - Anwendungsentwicklung
✓ [ 5/12] 15719-44156922-55-S          (4.5s)  Data Scientist in Advanced Analytics (m/
✓ [ 6/12] 16818-100787927-S            (5.2s)  Fachinformatiker*in Anwendungsentwicklun
✓ [ 7/12] 19384-938440348-S            (4.7s)  Data Scientist (m/w/d) – Pharmagroßhande
✓ [ 8/12] 10001-1003030981-S           (5.3s)  Fachinformatiker/in - Anwendungsentwickl
✓ [ 9/12] 16818-100814418-S            (4.8s)  Fachinformatiker*in Anwendungsentwicklun
✓ [10/12] 13644-297872-S               (5.2s)  Data Scientist (m/w/d)
✓ [11/12] 13319-868490/1_607417LS-S    (5.1s)  Data Scientist (m/w/d)
✓ [12/12] 10001-1002616378-S           (5.3s)  Data Scientist (m/w/d)

───────────────

In [7]:
"""
Schema-Validierung der Predictions mit dem mitgelieferten validate.py.
"""
import subprocess
result = subprocess.run(
    ["python", "../annotation/validate.py", "--validate-jsonl",
     "../daten/predictions.jsonl"],
    capture_output=True, text=True
)
print(result.stdout)
if result.stderr:
    print("STDERR:", result.stderr)


JSONL Schema-Check: predictions.jsonl
Geprüfte Zeilen: 12
JSON-Parse-Fails: 0

3 Feld-Verletzungen (sortiert):
     1×  homeoffice: leer
     1×  vertragsart: leer
     1×  erfahrungslevel: leer



In [8]:
"""
Welche Anzeige hat leere Pflichtfelder?
"""

print("Predictions-Übersicht:\n")
print(f"{'#':<3} {'refnr':<28} {'homeoffice':<16} {'vertragsart':<16} {'level':<16} {'gehalt':<8} {'skills'}")
print("─" * 130)

with open("../daten/predictions.jsonl", encoding="utf-8") as f:
    for i, line in enumerate(f, 1):
        p = json.loads(line)
        refnr = p.get("refnr", "?")
        ho = p.get("homeoffice", "")
        va = p.get("vertragsart", "")
        lvl = p.get("erfahrungslevel", "")
        ge = p.get("gehalt_min_eur", "")
        sk = p.get("skills_top3", "")
        
        # Markieren wenn ein Pflichtfeld leer ist
        warn = " ⚠" if (not ho or not va or not lvl) else ""
        print(f"{i:<3} {refnr:<28} {str(ho):<16} {str(va):<16} {str(lvl):<16} {str(ge):<8} {sk}{warn}")

Predictions-Übersicht:

#   refnr                        homeoffice       vertragsart      level            gehalt   skills
──────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────
1   13644-294294-S               nicht_genannt    festanstellung   mid              None     ['python', 'sql', 'r']
2   10001-1002678126-S           nicht_genannt    festanstellung   mid              None     ['python', 'machine-learning', 'scikit-learn']
3   10000-1185445644-S           nicht_genannt    festanstellung   senior           None     ['python', 'r', 'sql']
4   10001-1002026714-S           nicht_genannt    ausbildung       junior           None     ['java', 'c#', 'sql']
5   15719-44156922-55-S                                                                       ⚠
6   16818-100787927-S            nicht_genannt    ausbildung       junior           None     ['java', 'delphi', 'sql']
7   19384-938440348-S            teilweise 

In [9]:
"""
Truncation-Diagnose: Warum hat Allianz leere Pflichtfelder?
Hypothese: Text zu lang, Profil-Block nach Truncation verschwunden.
"""
allianz_text = korpus["15719-44156922-55-S"]["text"]
print(f"Allianz-Text-Länge: {len(allianz_text)} Zeichen")
print(f"Truncation-Limit:    {MAX_TEXT_LAENGE} Zeichen")
print(f"Abgeschnitten:       {len(allianz_text) - MAX_TEXT_LAENGE} Zeichen weg")
print(f"\n--- Letzte 300 Zeichen VOR dem Cut: ---")
print(allianz_text[MAX_TEXT_LAENGE-300:MAX_TEXT_LAENGE])
print(f"\n--- Erste 300 Zeichen DANACH (wurden weggeworfen): ---")
print(allianz_text[MAX_TEXT_LAENGE:MAX_TEXT_LAENGE+300])

Allianz-Text-Länge: 6917 Zeichen
Truncation-Limit:    1500 Zeichen
Abgeschnitten:       5417 Zeichen weg

--- Letzte 300 Zeichen VOR dem Cut: ---
 die Welt um uns herum.
Bei der Allianz stehen wir für Zusammenhalt: Wir glauben, dass eine geeinte Welt eine zukunftsfähige Welt ist, und setzen uns konsequent für Chancengleichheit für alle ein. Die Grundlage dafür ist unser inklusiver Arbeitsplatz, an dem sowohl Mensch als auch Leistung zählen un

--- Erste 300 Zeichen DANACH (wurden weggeworfen): ---
d eine Kultur gepflegt wird, die auf Integrität, Fairness, Inklusion und Vertrauen basiert.
Wir freuen uns über Bewerbungen von Menschen aller ethnischen und kulturellen Hintergründe, Altersgruppen, Geschlechter, Nationalitäten, Religionen, sozialen Herkünfte, Menschen mit oder ohne Behinderung sowi


## Truncation-Diagnose: Anzeige `15719-44156922-55-S` (Allianz)

Die Schema-Verletzungen aus `validate.py` (3× leeres Pflichtfeld) traten alle 
in dieser einen Anzeige auf. Diagnose:

| Kennzahl | Wert |
|---|---|
| Text-Länge gesamt | 6917 Zeichen |
| Truncation-Limit | 2000 Zeichen |
| Weggeworfen | 4917 Zeichen (71 % des Texts) |

**Was steht im sichtbaren Teil (Zeichen 0-2000)?**
Konzern-Boilerplate: Unternehmensvorstellung, Diversity-Statement, Slogans 
("Let's care for tomorrow"), Begrüßung. Kein Stelleninhalt.

**Was steht im weggeworfenen Teil (Zeichen 2000+)?**
- Konkrete Aufgaben (Problemlösung, Data Engineering, Modellierung)
- Anforderungen (Hochschulabschluss, Fachwissen, Praxiserfahrung)
- Skills (Python, SQL, scikit-learn, Cloud, CI/CD)
- Arbeitsmodell ("flexibel, hybrid, mobil und vor Ort")

→ Das Modell hat keine Grundlage für `homeoffice`, `vertragsart` oder 
`erfahrungslevel`, weil die relevanten Textstellen außerhalb des Kontexts liegen.

**Klassifikation: Pipeline-Problem (nicht Modell, nicht Schema).**

Das ist ein konkreter Iterations-Kandidat für Phase 4 — z.B.: 
- Truncation hochsetzen (kostet Inferenz-Zeit, eventuell OOM)
- Mittel-Truncation statt Head (Anfang + Ende behalten, Mitte droppen)
- Heuristik "Stellenanzeigen-Block finden" (suche nach `## Aufgaben`, 
  `## Profil`, etc.)

In [10]:
"""
=== PHASE 6 BLOCK 6.1: FINALE PIPELINE FÜR VOLLEN 7B-RUN ===

Finale Pipeline = Iteration B (Anti-Halluzinations-Regeln) + Bugfix für 
JSON-null-Codierung. Der Bugfix adressiert den Phase-4-Befund, dass Iter B 
zwar Halluzinationen reduziert, aber neue Halluzination einführt: Modell 
schreibt JSON-Wert `0` statt JSON-Schlüsselwort `null` bei Anzeigen ohne 
Gehaltsangabe.

Hinzugefügte Regel (unten in HARTE REGELN Punkt 1, präziser formuliert).
"""

SCHEMA_BESCHREIBUNG = """Du bist ein Assistent, der strukturierte Informationen aus deutschen Stellenanzeigen extrahiert.

Antworte AUSSCHLIESSLICH mit einem JSON-Objekt im folgenden Schema. Kein Vor- oder Nachtext.

SCHEMA (6 Felder, fix):
- "homeoffice": einer von ["ja", "teilweise", "nein", "remote", "nicht_genannt"]
- "vertragsart": einer von ["ausbildung", "festanstellung", "praktikum", "werkstudent", "sonstiges"]
- "erfahrungslevel": einer von ["junior", "mid", "senior", "egal", "nicht_genannt"]
- "gehalt_min_eur": ganze Zahl (int) oder JSON-Schlüsselwort null
- "gehalt_zeitraum": "monat", "jahr" oder null (muss null sein wenn gehalt_min_eur null ist)
- "skills_top3": Liste mit max. 3 technischen Skills/Tools/Sprachen

BEISPIEL-ANTWORT mit Zahl:
{"homeoffice": "teilweise", "vertragsart": "festanstellung", "erfahrungslevel": "mid", "gehalt_min_eur": 55000, "gehalt_zeitraum": "jahr", "skills_top3": ["python", "sql", "tableau"]}

BEISPIEL-ANTWORT ohne Zahl:
{"homeoffice": "nicht_genannt", "vertragsart": "festanstellung", "erfahrungslevel": "mid", "gehalt_min_eur": null, "gehalt_zeitraum": null, "skills_top3": ["python", "sql"]}

HARTE REGELN (BEFOLGE STRIKT):

1. gehalt_min_eur:
   - Wenn KEINE konkrete Zahl im Text steht: schreibe das JSON-Schlüsselwort null (lowercase, ohne Anführungszeichen).
   - NIEMALS die Zahl 0 schreiben. 0 bedeutet "Gehalt ist Null", null bedeutet "keine Angabe".
   - 'Attraktive Vergütung', 'nach Vereinbarung', 'marktüblich' → null.
   - Bei Range "ab 50.000 €" → 50000 (untere Grenze, ohne Punkt/Komma/Euro).

2. erfahrungslevel:
   - 'Ausbildung' / 'Auszubildende' → IMMER junior
   - 'erste Berufserfahrung' / 'max. 2 Jahre' / 'Berufseinsteiger' / 'Studienabsolvent' / '0-2 Jahre' → IMMER junior
   - 'Senior' im Titel ODER '5+ Jahre' / '8+ Jahre' / 'Mentoring von Juniors' → senior
   - '2-5 Jahre' / 'mehrjährig' / 'Erfahrung erforderlich' (ohne weitere Spezifikation) → mid
   - egal: NUR wenn explizit 'Junior bis Senior willkommen' o.ä. steht. Niemals als Default.
   - nicht_genannt: NUR wenn Text gar keine Aussage zur Erfahrung macht.

3. homeoffice:
   - 'hybrid' / 'X Tage/Woche Homeoffice' / 'mobiles Arbeiten' / 'Home Office möglich' → teilweise
   - '100% remote' / 'Vollzeit Remote' → remote
   - 'Präsenzpflicht' / 'kein Homeoffice' → nein
   - 'Homeoffice-Möglichkeit' OHNE Details → ja
   - Marketing-Slogans wie 'feels like home', 'familiäres Umfeld' → NICHT als Homeoffice werten. Das ist Werbung.
   - 'flexible Arbeitszeit' bezieht sich auf ZEIT, NICHT auf ORT → NICHT als Homeoffice werten.
   - Nichts dazu im Text → nicht_genannt.

4. skills_top3:
   - NUR konkrete Tools, Sprachen, Frameworks, Plattformen (Beispiele: Python, SQL, Java, Spark, AWS, Azure, scikit-learn, React, Power BI, Tableau, Git).
   - NICHT Methoden oder Konzepte: 'agile', 'machine learning', 'frameworks', 'testen', 'softwareentwicklung'.
   - NICHT Soft Skills: 'Teamfähigkeit', 'Kommunikation'.
   - NICHT Sprachen: 'Deutsch', 'Englisch'.
   - Bei Ausbildungs-Stellen: nur Tools, die explizit als BEWERBUNGSVORAUSSETZUNG genannt sind (z.B. 'Erste Programmierkenntnisse in Java erforderlich'). NICHT Tools, die 'gelernt werden' während der Ausbildung.
   - Wenn keine konkreten Tools im Text → leere Liste [].

5. gehalt_zeitraum: muss null sein, wenn gehalt_min_eur null ist.
"""

MAX_TEXT_LAENGE = 1500   # Wie Iter B — wegen GPU-Memory mit großem Prompt


def baue_prompt(anzeigentext: str) -> str:
    """Finale Pipeline: head-truncation 1500 + verschärfter Prompt mit null-Fix."""
    text_kurz = anzeigentext[:MAX_TEXT_LAENGE]
    messages = [
        {"role": "system", "content": SCHEMA_BESCHREIBUNG},
        {"role": "user", "content": f"Extrahiere die Felder aus dieser Stellenanzeige:\n\n{text_kurz}"},
    ]
    return tokenizer.apply_chat_template(
        messages, tokenize=False, add_generation_prompt=True
    )


print("✓ Finale Pipeline aktiviert (Phase 6 Block 6.1)")
print(f"  Truncation: Head, {MAX_TEXT_LAENGE} Zeichen")
print(f"  System-Prompt: {len(SCHEMA_BESCHREIBUNG)} Zeichen")
print(f"  Bugfix gegenüber Iter B: explizite null-vs-0-Regel + Beispiel ohne Zahl")

✓ Finale Pipeline aktiviert (Phase 6 Block 6.1)
  Truncation: Head, 1500 Zeichen
  System-Prompt: 3309 Zeichen
  Bugfix gegenüber Iter B: explizite null-vs-0-Regel + Beispiel ohne Zahl


In [17]:
"""
=== VOLLER KORPUS-RUN: alle 55 Anzeigen ===
Erwartete Dauer: ~5 Minuten (3-5s pro Anzeige).
"""

# Korpus laden (alle 55, nicht nur die 12 Gold)
korpus_pfad = Path("../daten/eigener_korpus.jsonl")
alle_anzeigen = []
with open(korpus_pfad, encoding="utf-8") as f:
    for line in f:
        alle_anzeigen.append(json.loads(line))

ausgabe_pfad = Path("../daten/predictions_7b_full.jsonl")
print(f"Starte vollen 7B-Run: {len(alle_anzeigen)} Anzeigen → {ausgabe_pfad}\n")

stats = {"ok": 0, "parse_fail": 0, "gesamt_dauer": 0.0}

with ausgabe_pfad.open("w", encoding="utf-8") as f_out:
    for i, anzeige in enumerate(alle_anzeigen, 1):
        refnr = anzeige["refnr"]
        parsed, raw, dauer = extrahiere(anzeige["text"])
        stats["gesamt_dauer"] += dauer

        if parsed is None:
            stats["parse_fail"] += 1
            output = {"refnr": refnr, "parse_fail": True, "raw_response": raw}
            status = "✗ PARSE-FAIL"
        else:
            stats["ok"] += 1
            output = {"refnr": refnr, **parsed}
            status = "✓"

        f_out.write(json.dumps(output, ensure_ascii=False) + "\n")
        f_out.flush()

        # Nur jeden 5. ausgeben, sonst zu viel Output
        if i % 5 == 0 or i == len(alle_anzeigen) or status.startswith("✗"):
            print(f"{status} [{i:>3}/{len(alle_anzeigen)}] {refnr:<28} "
                  f"({dauer:.1f}s)  {anzeige['titel'][:40]}")

print(f"\n{'─'*70}")
print(f"✓ Fertig! {stats['ok']} OK, {stats['parse_fail']} Parse-Fails")
print(f"  Gesamt-Dauer: {stats['gesamt_dauer']:.1f}s "
      f"(Ø {stats['gesamt_dauer']/len(alle_anzeigen):.1f}s pro Anzeige)")
print(f"  Output: {ausgabe_pfad}")

Starte vollen 7B-Run: 55 Anzeigen → ../daten/predictions_7b_full.jsonl



/opt/conda/envs/torch/lib/python3.10/site-packages/transformers/generation/configuration_utils.py:492: UserWarning: `do_sample` is set to `False`. However, `temperature` is set to `0.7` -- this flag is only used in sample-based generation modes. You should set `do_sample=True` or unset `temperature`.
  warnings.warn(
/opt/conda/envs/torch/lib/python3.10/site-packages/transformers/generation/configuration_utils.py:497: UserWarning: `do_sample` is set to `False`. However, `top_p` is set to `0.8` -- this flag is only used in sample-based generation modes. You should set `do_sample=True` or unset `top_p`.
  warnings.warn(
/opt/conda/envs/torch/lib/python3.10/site-packages/transformers/generation/configuration_utils.py:509: UserWarning: `do_sample` is set to `False`. However, `top_k` is set to `20` -- this flag is only used in sample-based generation modes. You should set `do_sample=True` or unset `top_k`.
  warnings.warn(


RuntimeError: CUDA out of memory. Tried to allocate 246.00 MiB (GPU 0; 31.75 GiB total capacity; 30.12 GiB already allocated; 514.88 MiB free; 30.38 GiB reserved in total by PyTorch) If reserved memory is >> allocated memory try setting max_split_size_mb to avoid fragmentation.  See documentation for Memory Management and PYTORCH_CUDA_ALLOC_CONF

In [16]:
"""
Schema-Validierung des vollen 7B-Runs.
Auf den 12 Hand-Gold-Anzeigen kommt später Per-Field-Accuracy (in 03_eval.ipynb).
Auf den 43 Anzeigen ohne Gold: Schema-Konformität ist alles, was wir prüfen können.
"""
import subprocess
result = subprocess.run(
    ["python", "../annotation/validate.py", "--validate-jsonl",
     "../daten/predictions_7b_full.jsonl"],
    capture_output=True, text=True
)
print(result.stdout)
if result.stderr:
    print("STDERR:", result.stderr)


JSONL Schema-Check: predictions_7b_full.jsonl
Geprüfte Zeilen: 0
JSON-Parse-Fails: 0
Keine Feld-Verletzungen.

